# Binary Classification: Model Evaluation Beyond Accuracy

**Type:** Original supplementary practice  
**Data:** Synthetic classification dataset

This notebook trains logistic regression and random forest classifiers, uses a stratified holdout, and compares precision, recall, F1, ROC-AUC and confusion matrices. The classes are simulated and do not represent a real risk or eligibility decision.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, ConfusionMatrixDisplay, classification_report,
    RocCurveDisplay
)

X, y = make_classification(
    n_samples=2400, n_features=12, n_informative=7, n_redundant=2,
    weights=[.78,.22], class_sep=1.0, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.25, random_state=42, stratify=y
)
print("Train class proportions:", np.bincount(y_train)/len(y_train))
print("Test class proportions:", np.bincount(y_test)/len(y_test))

## 1. Fit two models

In [ ]:
models = {
    "Logistic regression": Pipeline([
        ("scale", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ]),
    "Random forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=3,
        class_weight="balanced", random_state=42, n_jobs=-1
    )
}
for model in models.values():
    model.fit(X_train, y_train)

## 2. Compare metrics

In [ ]:
rows = []
for name, model in models.items():
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:,1]
    rows.append({
        "model":name,
        "accuracy":accuracy_score(y_test,pred),
        "precision":precision_score(y_test,pred,zero_division=0),
        "recall":recall_score(y_test,pred,zero_division=0),
        "f1":f1_score(y_test,pred,zero_division=0),
        "roc_auc":roc_auc_score(y_test,prob)
    })
metrics = pd.DataFrame(rows).set_index("model")
display(metrics.round(3))

## 3. Inspect the operating point

In [ ]:
chosen = models["Logistic regression"]
pred = chosen.predict(X_test)
print(classification_report(y_test, pred, target_names=["Class 0","Class 1"]))
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title("Logistic regression: confusion matrix")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6,5))
for name, model in models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)
ax.plot([0,1],[0,1], linestyle="--", linewidth=1)
ax.set_title("ROC curves")
plt.tight_layout()
plt.show()

## Interpretation

Accuracy can conceal poor minority-class performance on imbalanced data. Precision and recall reflect different error costs. The threshold of 0.5 is a default, not a universal optimum; choose a threshold using validation data and the consequences of false positives and false negatives.

**Limitations:** This is a simulated classification task. A real application requires data provenance, leakage checks, subgroup performance analysis, calibration, an appropriate threshold policy and monitoring.